# Kaggle 30M — continue fit (AdamW + L2 0.1, dropout 0.05, 20 epochs)

**Settings → Accelerator → GPU T4** (not P100). Restart, then run cell 1 and cell 2 separately.
Cell 1 copies code + `last.pt`. Cell 2 resumes. Do not raise weight decay.

In [ ]:
from pathlib import Path
import shutil, zipfile, os, sys, torch

assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4, Restart session."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(name, "sm", major, minor)
assert major >= 7, (
    f"{name} is Pascal (P100). This Kaggle PyTorch cannot run it. "
    "Settings → Accelerator → GPU T4 → Restart session."
)

INP = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN = WORK / "run"

def first(name):
    hits = list(INP.rglob(name))
    return hits[0] if hits else None

train_py = [p for p in INP.rglob("scripts/train.py") if "__MACOSX" not in str(p)]
if train_py:
    src_root = train_py[0].parent.parent
else:
    z = first("src30m.zip") or first("ttt-code.zip") or first("colab_upload.zip")
    assert z, "Add Input: dataset with src30m.zip"
    tmp = WORK / "from_code_zip"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    with zipfile.ZipFile(z) as zf:
        zf.extractall(tmp)
    src_root = next(p.parent.parent for p in tmp.rglob("scripts/train.py") if "__MACOSX" not in str(p))

if RUN.exists():
    shutil.rmtree(RUN)
shutil.copytree(src_root, RUN, ignore=shutil.ignore_patterns("*.pyc", "__pycache__"))
os.chdir(RUN)
sys.path.insert(0, str(RUN))
print("working dir", Path.cwd())
print("has ckpt.py", (Path("tttmem") / "ckpt.py").exists())
print("train.py bytes", Path("scripts/train.py").stat().st_size)

out = WORK / "checkpoints" / "baseline-30m"
out.mkdir(parents=True, exist_ok=True)
dest = out / "last.pt"

ckpts = [p for p in INP.rglob("last.pt") if p.is_file() and p.stat().st_size > 50_000_000]
if not ckpts:
    existing = WORK / "last.pt"
    if existing.is_file() and existing.stat().st_size > 50_000_000:
        ckpts = [existing]
assert ckpts, "Need last.pt on the dataset or already in /kaggle/working"
src_ckpt = max(ckpts, key=lambda p: p.stat().st_size)
if src_ckpt.resolve() != dest.resolve():
    shutil.copy2(src_ckpt, dest)
print("copied file checkpoint", src_ckpt, "MB", round(src_ckpt.stat().st_size / 1e6, 1))
shutil.copy2(dest, WORK / "last.pt")
print("READY")

INP = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN = WORK / "run"

def first(name):
    hits = list(INP.rglob(name))
    return hits[0] if hits else None

train_py = [p for p in INP.rglob("scripts/train.py") if "__MACOSX" not in str(p)]
if train_py:
    src_root = train_py[0].parent.parent
else:
    z = first("src30m.zip") or first("ttt-code.zip") or first("colab_upload.zip")
    assert z, "Add Input: dataset with src30m.zip (re-zip kaggle_upload/src30m, not the 1 Sep zip)"
    tmp = WORK / "from_code_zip"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    with zipfile.ZipFile(z) as zf:
        zf.extractall(tmp)
    src_root = next(p.parent.parent for p in tmp.rglob("scripts/train.py") if "__MACOSX" not in str(p))

if RUN.exists():
    shutil.rmtree(RUN)
shutil.copytree(src_root, RUN, ignore=shutil.ignore_patterns("*.pyc", "__pycache__"))
os.chdir(RUN)
sys.path.insert(0, str(RUN))
print("working dir", Path.cwd())
print("has ckpt.py", (Path("tttmem") / "ckpt.py").exists())
print("train.py bytes", Path("scripts/train.py").stat().st_size)

out = WORK / "checkpoints" / "baseline-30m"
out.mkdir(parents=True, exist_ok=True)
dest = out / "last.pt"

ckpts = [p for p in INP.rglob("last.pt") if p.is_file() and p.stat().st_size > 50_000_000]
if not ckpts:
    existing = WORK / "last.pt"
    if existing.is_file() and existing.stat().st_size > 50_000_000:
        ckpts = [existing]
if ckpts:
    src_ckpt = max(ckpts, key=lambda p: p.stat().st_size)
    if src_ckpt.resolve() != dest.resolve():
        shutil.copy2(src_ckpt, dest)
    print("copied file checkpoint", src_ckpt, "MB", round(src_ckpt.stat().st_size / 1e6, 1))
else:
    pkls = [p for p in INP.rglob("data.pkl") if "last" in str(p)]
    assert pkls or first("checkpoints.zip"), "Need last.pt on the dataset"
    if not pkls:
        destk = WORK / "from_ckpt_zip"
        if destk.exists():
            shutil.rmtree(destk)
        destk.mkdir()
        with zipfile.ZipFile(first("checkpoints.zip")) as zf:
            zf.extractall(destk)
        pkls = list(destk.rglob("data.pkl"))
        file_ckpts = [p for p in destk.rglob("last.pt") if p.is_file()]
        if file_ckpts:
            shutil.copy2(max(file_ckpts, key=lambda p: p.stat().st_size), dest)
            pkls = []
    if pkls:
        from tttmem.ckpt import pack_unpacked_ckpt, resolve_ckpt
        folder = resolve_ckpt(pkls[0].parent)
        pack_unpacked_ckpt(folder, dest)
        print("packed unzipped checkpoint", folder)

shutil.copy2(dest, WORK / "last.pt")
print("READY")
print("DOWNLOAD NOW (refresh Output): /kaggle/working/last.pt  MB",
      round((WORK / "last.pt").stat().st_size / 1e6, 1))

In [ ]:
%pip install -q tokenizers numpy
from pathlib import Path
import os

os.chdir("/kaggle/working/run")
out = Path("/kaggle/working/checkpoints/baseline-30m")
resume = out / "last.pt"
assert resume.exists() and resume.stat().st_size > 50_000_000
assert Path("scripts/train.py").exists()

# Same AdamW L2=0.1; lower dropout + more epochs so val PPL can keep falling.
!python -u scripts/train.py --epochs 20 --optimizer adamw --weight-decay 0.1 \
  --dropout 0.05 --min-lr-ratio 0.1 --beta2 0.95 \
  --device cuda --amp auto --seq-len 256 --batch-size 2 --grad-accum 2 \
  --data data/tinystories-v2/train.npy --val-data data/tinystories-v2/validation.npy \
  --out {out} --resume {resume}

print("done listing:")
!ls -lh /kaggle/working/last.pt /kaggle/working/checkpoints/baseline-30m/